# AlphaForge — Feature Analysis
Signal quality analysis: Information Coefficient, feature importance, autocorrelation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from src.data.storage import load_features
from src.features.technical import FEATURE_COLUMNS
from src.config import get_settings

settings = get_settings()
asset = 'BTC/USDT'
df = load_features(asset, '1h')
print(f'Loaded {len(df):,} feature rows for {asset}')
df[FEATURE_COLUMNS].describe().round(4)

In [ ]:
# ── Information Coefficient per feature ──────────────────────────
# IC = Spearman rank correlation between feature at t and forward return at t+h
# IC > 0.03 is considered a good alpha signal

results = []
for feat in FEATURE_COLUMNS:
    if feat not in df.columns:
        continue
    for h in [1, 5, 20]:
        label = f'fwd_ret_{h}'
        if label not in df.columns:
            continue
        sub = df[[feat, label]].dropna()
        if len(sub) < 50:
            continue
        ic, p = spearmanr(sub[feat], sub[label])
        results.append({'feature': feat, 'horizon': h, 'ic': ic, 'p_value': p})

ic_df = pd.DataFrame(results)
ic_pivot = ic_df.pivot(index='feature', columns='horizon', values='ic')
ic_pivot.columns = [f'IC_h{c}' for c in ic_pivot.columns]
ic_pivot['mean_ic'] = ic_pivot.mean(axis=1)
ic_pivot = ic_pivot.sort_values('mean_ic', ascending=False)
print('Top 10 features by mean IC:')
print(ic_pivot.head(10).round(4))

In [ ]:
# IC heatmap
fig, ax = plt.subplots(figsize=(10, 12))
data = ic_pivot[['IC_h1', 'IC_h5', 'IC_h20']].values
im = ax.imshow(data, cmap='RdYlGn', vmin=-0.1, vmax=0.1, aspect='auto')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Horizon 1', 'Horizon 5', 'Horizon 20'])
ax.set_yticks(range(len(ic_pivot)))
ax.set_yticklabels(ic_pivot.index, fontsize=9)
plt.colorbar(im, ax=ax, label='Information Coefficient')
ax.set_title('Feature Information Coefficient Heatmap\n(Green = positive predictive power)')
plt.tight_layout()
plt.show()

In [ ]:
# ── RSI signal analysis ───────────────────────────────────────────
# Quintile returns: do RSI quintiles predict returns?
df_clean = df[['rsi_14', 'fwd_ret_1']].dropna()
df_clean['rsi_quintile'] = pd.qcut(df_clean['rsi_14'], q=5, labels=[1,2,3,4,5])
quintile_returns = df_clean.groupby('rsi_quintile')['fwd_ret_1'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ef4444', '#f97316', '#eab308', '#84cc16', '#22c55e']
bars = ax.bar(quintile_returns.index, quintile_returns.values, color=colors)
ax.axhline(0, color='white', linewidth=0.5)
ax.set_xlabel('RSI Quintile (1=Oversold, 5=Overbought)')
ax.set_ylabel('Mean 1-Bar Forward Return (%)')
ax.set_title('RSI Quintile → Forward Return Analysis')
ax.set_facecolor('#0b1020')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature autocorrelation (do features persist?) ────────────────
top_features = ic_pivot.head(8).index.tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    if feat in df.columns:
        series = df[feat].dropna()
        autocorrs = [series.autocorr(lag=lag) for lag in range(1, 25)]
        axes[i].bar(range(1, 25), autocorrs, color='#00d4ff', alpha=0.7)
        axes[i].axhline(0, color='white', linewidth=0.5)
        axes[i].set_title(feat, fontsize=9)
        axes[i].set_facecolor('#0b1020')

plt.suptitle('Feature Autocorrelation (lags 1-24)')
plt.tight_layout()
plt.show()